# FS03-04 captioning (nan-safe CE)


In [ ]:
import os, json, math, random, time
from pathlib import Path
import numpy as np
os.environ.pop('CUDA_VISIBLE_DEVICES', None)
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT=Path('/kaggle/working'); FIG=OUT/'figures'; RES=OUT/'results'
FIG.mkdir(parents=True, exist_ok=True); RES.mkdir(parents=True, exist_ok=True)
device=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device',device,'gpus',torch.cuda.device_count() if torch.cuda.is_available() else 0)
PROGRESS={}; GATES={}
def gate(name, ok, detail=''):
    GATES[name]=bool(ok); print(('PASS' if ok else 'FAIL'), name, detail)
    if not ok: raise AssertionError(f'ACCEPTANCE FAILED: {name} {detail}')
def make_shape_image(kind, size=64):
    img=np.ones((size,size,3),np.float32)*0.95
    yy,xx=np.mgrid[0:size,0:size]; cy,cx=size//2,size//2
    if kind=='red_circle':
        m=(yy-cy)**2+(xx-cx)**2<=(size*0.28)**2; img[m]=(0.9,0.15,0.12)
    elif kind=='blue_square':
        m=(np.abs(yy-cy)<size*0.25)&(np.abs(xx-cx)<size*0.25); img[m]=(0.15,0.25,0.85)
    elif kind=='green_triangle':
        top=cy-int(size*0.28); bot=cy+int(size*0.30)
        for y in range(max(0,top),min(size,bot)):
            half=int((y-top)/max(1,bot-top)*size*0.30)
            img[y, max(0,cx-half):min(size,cx+half+1)]=(0.15,0.75,0.25)
    else: raise ValueError(kind)
    return img
CLASSES=['red_circle','blue_square','green_triangle']
c2i={c:i for i,c in enumerate(CLASSES)}
CAPTIONS={c:'a '+c.replace('_',' ') for c in CLASSES}


## FS03


In [ ]:
PAD,BOS,EOS='<pad>','<bos>','<eos>'
vocab=[PAD,BOS,EOS]+sorted({w for c in CAPTIONS.values() for w in c.split()})
stoi={t:i for i,t in enumerate(vocab)}; itos=vocab; V=len(vocab); MAXL=6

def enc_cap(s):
    ids=[stoi[BOS]]+[stoi[w] for w in s.split()]+[stoi[EOS]]
    return (ids+[stoi[PAD]]*MAXL)[:MAXL]

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),
            nn.Flatten(), nn.Linear(128,3))
    def forward(self,x): return self.net(x)

class CapLSTM(nn.Module):
    def __init__(self,d=64,h=128):
        super().__init__()
        self.cls_emb=nn.Embedding(3,d)
        self.tok=nn.Embedding(V,d,padding_idx=stoi[PAD])
        self.lstm=nn.LSTM(d,h,batch_first=True)
        self.fc=nn.Linear(h,V)
        self.h0=nn.Linear(d,h)
    def forward(self, cls, caps):
        e0=self.cls_emb(cls)
        h=torch.tanh(self.h0(e0)).unsqueeze(0); c=torch.zeros_like(h)
        te=self.tok(caps[:,:-1]).clone(); te[:,0]=te[:,0]+e0
        o,_=self.lstm(te,(h,c)); return self.fc(o)

def make_data(n_per=300,size=32):
    xs,ys,cs=[],[],[]
    for k in CLASSES:
        for _ in range(n_per):
            img=make_shape_image(k,64)[::2,::2][:size,:size]
            img=np.clip(img+0.04*np.random.randn(*img.shape).astype(np.float32),0,1)
            xs.append(img.transpose(2,0,1)); ys.append(enc_cap(CAPTIONS[k])); cs.append(c2i[k])
    return torch.tensor(np.stack(xs),dtype=torch.float32), torch.tensor(ys), torch.tensor(cs)

X,Y,C=make_data(); perm=torch.randperm(len(X)); X,Y,C=X[perm],Y[perm],C[perm]
ntr=int(0.85*len(X)); Xtr,Ytr,Ctr=X[:ntr],Y[:ntr],C[:ntr]; Xva,Yva,Cva=X[ntr:],Y[ntr:],C[ntr:]
cnn=CNN().to(device); cap=CapLSTM().to(device)
opt=torch.optim.Adam(list(cnn.parameters())+list(cap.parameters()), lr=1e-3)
hist=[]
for epoch in range(1,25):
    cnn.train(); cap.train(); losses=[]; cacc=0; n=0
    for i in range(0,len(Xtr),64):
        xb,yb,cb=Xtr[i:i+64].to(device),Ytr[i:i+64].to(device),Ctr[i:i+64].to(device)
        opt.zero_grad(set_to_none=True)
        logits_c=cnn(xb)
        loss_c=F.cross_entropy(logits_c, cb)
        loss_l=F.cross_entropy(cap(cb,yb).reshape(-1,V), yb[:,1:].reshape(-1), ignore_index=stoi[PAD])
        pred_c=logits_c.argmax(1).detach()
        loss_l2=F.cross_entropy(cap(pred_c,yb).reshape(-1,V), yb[:,1:].reshape(-1), ignore_index=stoi[PAD])
        loss=loss_c+loss_l+0.5*loss_l2
        loss.backward(); opt.step(); losses.append(float(loss.detach().cpu()))
        cacc+=(logits_c.argmax(1)==cb).sum().item(); n+=cb.size(0)
    cnn.eval(); cap.eval()
    with torch.no_grad():
        lg=cnn(Xva.to(device)); vacc=(lg.argmax(1)==Cva.to(device)).float().mean().item()
    hist.append({'epoch':epoch,'loss':round(float(np.mean(losses)),4),'cls_train':round(cacc/n,4),'cls_val':round(vacc,4)})
    if epoch%5==0: print(hist[-1])

@torch.no_grad()
def caption(kind):
    img=make_shape_image(kind,64)[::2,::2][:32,:32]
    x=torch.tensor(img.transpose(2,0,1)[None],dtype=torch.float32,device=device)
    cnn.eval(); cap.eval()
    cls=cnn(x).argmax(1)
    e0=cap.cls_emb(cls); h=torch.tanh(cap.h0(e0)).unsqueeze(0); c=torch.zeros_like(h)
    tok=torch.tensor([[stoi[BOS]]],device=device); out=[]
    for t in range(MAXL-1):
        te=cap.tok(tok[:,-1:])
        if t==0: te=te+e0.unsqueeze(1)
        o,(h,c)=cap.lstm(te,(h,c))
        nxt=cap.fc(o[:,-1]).argmax(-1)
        if nxt.item()==stoi[EOS]: break
        if nxt.item()!=stoi[PAD]: out.append(itos[nxt.item()])
        tok=torch.cat([tok,nxt[:,None]],1)
    return ' '.join(out), CLASSES[cls.item()]

samples=[]
for k in CLASSES:
    pred,cls=caption(k)
    samples.append({'input':k,'pred_cls':cls,'pred':pred,'gt':CAPTIONS[k],'ok':pred==CAPTIONS[k]})
    print(samples[-1])
exact=sum(s['ok'] for s in samples)/len(samples)
gate('FS03_cls_val', hist[-1]['cls_val']>=0.95, hist[-1])
gate('FS03_greedy_exact', exact>=0.999, samples)
gate('FS03_loss_finite', all(math.isfinite(h['loss']) for h in hist), hist[-1])
fig,axes=plt.subplots(1,3,figsize=(9,3))
for ax,s in zip(axes,samples):
    ax.imshow(make_shape_image(s['input'])); ax.set_title(s['pred'],fontsize=9); ax.axis('off')
fig.tight_layout(); fig.savefig(FIG/'fs03_captions.png',dpi=120); plt.close()
(RES/'fs03.json').write_text(json.dumps({'stage':'FS03','method':'ShowTell CNN class-bottleneck + LSTM','history':hist,'samples':samples,'greedy_exact':exact,'vs_prev':'class word -> sequence caption'},indent=2))
PROGRESS['FS03']='ok'


## FS04


In [ ]:
# Spatial attention captioner — CE with explicit non-pad mask (no all-pad nan)
class AttnCap(nn.Module):
    def __init__(self,d=64,h=128):
        super().__init__()
        self.cnn=nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,d,3,padding=1),nn.ReLU())
        self.pool=nn.AdaptiveAvgPool2d(1)
        self.cls=nn.Linear(d,3)
        self.tok=nn.Embedding(V,d,padding_idx=stoi[PAD])
        self.attn=nn.Linear(d+h,1)
        self.lstm=nn.LSTMCell(d+d,h)
        self.fc=nn.Linear(h,V)
        self.h0=nn.Linear(d,h)
    def encode(self,x):
        f=self.cnn(x)
        g=self.pool(f).flatten(1)
        feats=f.flatten(2).transpose(1,2)
        return feats, g, self.cls(g)
    def step(self, token, h, c, feats):
        B,L,D=feats.shape
        e=self.attn(torch.cat([feats,h.unsqueeze(1).expand(-1,L,-1)],-1)).squeeze(-1)
        a=torch.softmax(e,-1)
        ctx=(a.unsqueeze(-1)*feats).sum(1)
        h2,c2=self.lstm(torch.cat([self.tok(token),ctx],-1),(h,c))
        return self.fc(h2),h2,c2,a

def masked_ce(logits, target, ignore):
    """Cross-entropy that returns 0 (not nan) when all targets are ignore."""
    mask = target != ignore
    if mask.sum() == 0:
        return logits.sum() * 0.0
    return F.cross_entropy(logits[mask], target[mask])

am=AttnCap().to(device); opt=torch.optim.Adam(am.parameters(),lr=5e-4)
hist4=[]
for epoch in range(1,25):
    am.train(); losses=[]
    for i in range(0,len(Xtr),64):
        xb,yb,cb=Xtr[i:i+64].to(device),Ytr[i:i+64].to(device),Ctr[i:i+64].to(device)
        opt.zero_grad(set_to_none=True)
        feats,g,cl=am.encode(xb)
        loss=F.cross_entropy(cl,cb)
        h=torch.tanh(am.h0(g)); cstate=torch.zeros_like(h)
        tok_losses=[]
        for t in range(yb.size(1)-1):
            lg,h,cstate,a=am.step(yb[:,t],h,cstate,feats)
            tok_losses.append(masked_ce(lg, yb[:,t+1], stoi[PAD]))
        loss=loss+torch.stack(tok_losses).mean()
        if not torch.isfinite(loss):
            print('non-finite loss at epoch', epoch, 'batch', i); break
        loss.backward()
        torch.nn.utils.clip_grad_norm_(am.parameters(), 1.0)
        opt.step(); losses.append(float(loss.detach().cpu()))
    am.eval()
    with torch.no_grad():
        _,_,cl=am.encode(Xva.to(device)); vacc=(cl.argmax(1)==Cva.to(device)).float().mean().item()
    mean_loss=float(np.mean(losses)) if losses else float('nan')
    hist4.append({'epoch':epoch,'loss':round(mean_loss,4),'cls_val':round(vacc,4)})
    if epoch%5==0: print('FS04',hist4[-1])

@torch.no_grad()
def caption_attn(kind):
    img=make_shape_image(kind,64)[::2,::2][:32,:32]
    x=torch.tensor(img.transpose(2,0,1)[None],dtype=torch.float32,device=device)
    am.eval(); feats,g,cl=am.encode(x)
    h=torch.tanh(am.h0(g)); cstate=torch.zeros_like(h)
    tok=torch.tensor([stoi[BOS]],device=device); out=[]; maps=[]
    for t in range(MAXL-1):
        lg,h,cstate,a=am.step(tok,h,cstate,feats); maps.append(a.cpu().numpy()[0])
        nxt=lg.argmax(-1)
        if nxt.item()==stoi[EOS]: break
        if nxt.item()!=stoi[PAD]: out.append(itos[nxt.item()])
        tok=nxt
    return ' '.join(out), maps, CLASSES[cl.argmax(1).item()]

samples4=[]; fig,axes=plt.subplots(2,3,figsize=(9,5))
for j,k in enumerate(CLASSES):
    pred,maps,cls=caption_attn(k)
    samples4.append({'input':k,'pred':pred,'gt':CAPTIONS[k],'ok':pred==CAPTIONS[k],'cls':cls})
    axes[0,j].imshow(make_shape_image(k)); axes[0,j].set_title(pred,fontsize=8); axes[0,j].axis('off')
    if maps:
        a=maps[min(1,len(maps)-1)]; side=int(round(math.sqrt(len(a))))
        axes[1,j].imshow(make_shape_image(k)); axes[1,j].imshow(a.reshape(side,side),extent=[0,64,64,0],alpha=0.55,cmap='hot'); axes[1,j].axis('off')
fig.tight_layout(); fig.savefig(FIG/'fs04_attention.png',dpi=120); plt.close()
exact4=sum(s['ok'] for s in samples4)/len(samples4)
print(samples4)
gate('FS04_loss_finite', all(math.isfinite(h['loss']) for h in hist4), hist4[-1])
gate('FS04_cls_val', hist4[-1]['cls_val']>=0.95, hist4[-1])
gate('FS04_greedy_exact', exact4>=0.999, samples4)
(RES/'fs04.json').write_text(json.dumps({'stage':'FS04','method':'Attend caption spatial soft-attn','history':hist4,'samples':samples4,'greedy_exact':exact4,'vs_prev':'global->spatial attention'},indent=2))
PROGRESS['FS04']='ok'


In [ ]:
(RES/'summary_fs03_fs04.json').write_text(json.dumps({'progress':PROGRESS,'gates':GATES},indent=2))
(OUT/'SUCCESS').write_text('ok\n')
(OUT/'ACCEPTANCE.json').write_text(json.dumps({'ok':all(GATES.values()),'gates':GATES},indent=2))
print('FS03-04 PASS',GATES)
